In [2]:
!pip install tensorflow scikit-learn numpy

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

# 1. Synthetic Data Generation
np.random.seed(42)
num_samples = 2000

normal = np.random.normal(loc=[0.2, 0.1, 55.0, 24.0], scale=[0.05, 0.02, 5.0, 1.0], size=(num_samples // 3, 4))
wheezing = np.random.normal(loc=[0.85, 0.7, 50.0, 25.0], scale=[0.1, 0.1, 5.0, 1.0], size=(num_samples // 3, 4))
apnea = np.random.normal(loc=[0.02, 0.01, 35.0, 23.0], scale=[0.01, 0.01, 3.0, 1.0], size=(num_samples // 3, 4))

X = np.vstack([normal, wheezing, apnea])
y = np.array([0]*(num_samples // 3) + [1]*(num_samples // 3) + [2]*(num_samples // 3))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Train Lightweight MLP (Multi-Layer Perceptron)
clf = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=500, random_state=42)
clf.fit(X_train, y_train)

print(f"Model Accuracy: {clf.score(X_test, y_test) * 100:.2f}%")

Model Accuracy: 99.50%


C:\Users\BIT\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [4]:
# Extract weights and biases from the trained Scikit-learn MLP
w1 = clf.coefs_[0].tolist()       # Shape: (4, 16)
b1 = clf.intercepts_[0].tolist()   # Shape: (16,)
w2 = clf.coefs_[1].tolist()       # Shape: (16, 8)
b2 = clf.intercepts_[1].tolist()   # Shape: (8,)
w3 = clf.coefs_[2].tolist()       # Shape: (8, 3)
b3 = clf.intercepts_[2].tolist()   # Shape: (3,)

def fmt_matrix(mat):
    return "{\n  " + ",\n  ".join(["{" + ", ".join([f"{v:.6f}f" for v in row]) + "}" for row in mat]) + "\n}"

def fmt_vec(vec):
    return "{" + ", ".join([f"{v:.6f}f" for v in vec]) + "}"

c_code = f"""#ifndef MODEL_WEIGHTS_H
#ifndef MODEL_WEIGHTS_H
#define MODEL_WEIGHTS_H

// Input features: [Audio_RMS, ZCR, Humidity, Temp]
// Hidden Layer 1: 16 neurons (ReLU)
// Hidden Layer 2: 8 neurons (ReLU)
// Output Layer: 3 neurons (Softmax -> 0: Normal, 1: Wheezing, 2: Apnea)

static const float W1[4][16] = {fmt_matrix(w1)};
static const float B1[16] = {fmt_vec(b1)};

static const float W2[16][8] = {fmt_matrix(w2)};
static const float B2[8] = {fmt_vec(b2)};

static const float W3[8][3] = {fmt_matrix(w3)};
static const float B3[3] = {fmt_vec(b3)};

#endif // MODEL_WEIGHTS_H
"""

with open("model_weights.h", "w") as f:
    f.write(c_code)

print("Saved model_weights.h successfully!")

Saved model_weights.h successfully!
